# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SanniyaAslam/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "/content/flyrank-ml-internship"

if IN_COLAB and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/SanniyaAslam/flyrank-ml-internship.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub", "scikit-learn"], check=True)

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Working dir:", os.getcwd())

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

df = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_fh,
               SUM(gsc_clicks) AS clicks_fh,
               AVG(gsc_avg_position) AS avg_position_fh
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND EXTRACT(day FROM report_date) <= 15
        GROUP BY client_hash_id, content_hash_id
    ),
    second_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) AS clicks_sh
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND EXTRACT(day FROM report_date) > 15
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        fh.client_hash_id, fh.content_hash_id,
        fh.impressions_fh, fh.clicks_fh, fh.avg_position_fh,
        sh.clicks_sh,
        d.content_updated_date, d.word_count,
        DATE_DIFF('day', d.content_updated_date, DATE '2026-03-15') AS days_since_update_fh
    FROM first_half fh
    JOIN second_half sh ON fh.client_hash_id = sh.client_hash_id AND fh.content_hash_id = sh.content_hash_id
    JOIN read_parquet('{REL}/dim_content.parquet') d
      ON fh.client_hash_id = d.client_hash_id AND fh.content_hash_id = d.content_hash_id
    WHERE fh.impressions_fh > 0
""").df()

df["ctr_fh"] = df["clicks_fh"] / df["impressions_fh"]
df["is_declining"] = (df["clicks_sh"] < df["clicks_fh"]).astype(int)
df["word_count"] = df["word_count"].fillna(df["word_count"].median())
df["days_since_update_fh"] = df["days_since_update_fh"].clip(lower=0)

features = ["impressions_fh", "ctr_fh", "avg_position_fh", "days_since_update_fh", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining"].values
groups = df["client_hash_id"].values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print("Rows:", len(df))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*Finding #4 — The Freshness Multiplier (3.2x health boost, 57x impression
boost from refreshing 365+ day content):

Where does the label come from? The "growing vs declining" split behind
this finding comes from 30-day-vs-previous-30-day impression change
(Up: >10%, Down: >10% decline). The refresh comparison itself (365+ day
content refreshed within 30 days: 10.7 → 34.5 health, 71 → 4039
impressions) sits inside the paper's own "local active-content sample,"
and the paper itself flags that the neighboring 361+ freshness bucket
(283:1 growth ratio) is unstable because it has only 1 declining page.
My methodology question: is the headline 3.2x/57x refresh comparison
built from a similarly small, survivor-biased slice of the active-content
sample, or from a larger population? The paper doesn't state the sample
size for that specific before/after comparison, so it's hard to tell if
it sits on the same fragile footing as the 361+ bucket right next to it
in the same section.

Does the validation design carry the claim? This is a before/after
observational comparison, not a controlled or held-out test — pages that
got refreshed may differ systematically from pages that didn't (e.g.
already-valuable pages are more likely to get prioritized for refresh in
the first place). The paper is appropriately careful elsewhere about
survivorship bias (explicitly calling it out for the 361+ bucket), so my
question is simply whether the same caveat should extend to this
particular headline stat, since selection-into-refresh could inflate the
apparent effect size beyond what refreshing alone would cause.

---

ML Appendix — Growth Prediction (Logistic Regression, 71% holdout
accuracy):

Where does the label come from? The paper doesn't fully specify how
"growing" vs "declining" is labeled for this model — presumably the same
30d-vs-prev-30d rule used elsewhere in the paper, but this isn't restated
in the ML appendix section itself, so the exact label construction has to
be inferred from earlier sections.

Does the validation design carry the claim? The methodology section
states an 80/20 split for the Random Forest, Logistic Regression, and
Decision Tree models, but doesn't say whether this split is grouped by
client/brand or a simple random row split. Since the study spans 57
brands, a random split could let content from the same brand appear in
both train and test, letting the model partly learn brand-specific
baseline behavior rather than a generalizable growth signal — the same
concern my own Week-5/Week-6 work addressed with a client-grouped split.
My methodology question, asked in the same spirit the paper itself
applies to its own weaker constructs: was this 71% accuracy checked
against a brand-held-out split, or could some of it reflect brand-level
memorization?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Two paper findings and methodology questions documented above — no query needed for this section (this section reviews the FlyRank public paper, not my own warehouse data).")


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*Before (naive random split): Precision@50 = 0.74, ROC AUC = 0.928, with
41 clients appearing in BOTH train and test.

After (client-grouped split): Precision@50 = 0.74, ROC AUC = 0.907, with
0 client overlap.

Interestingly, Precision@50 stayed identical here, but ROC AUC dropped
slightly (0.928 → 0.907) once client overlap was removed — a small but
real sign that the naive split was getting a bit of an unearned boost
from seeing some of each overlapping client's other pages during
training. The gap isn't dramatic in this case, but the 41-client overlap
itself is the real problem: it means my earlier "honest" number was
never fully honest, even though the final precision score barely moved.
I trust the grouped-split number (0.907 AUC) as the one that reflects
performance on genuinely unseen clients.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.# BEFORE — naive random split (rows, ignoring that a client can appear in both train & test)
X_train_naive, X_test_naive, y_train_naive, y_test_naive, idx_train_naive, idx_test_naive = train_test_split(
    X, y, df.index, test_size=0.25, random_state=42, stratify=y
)
overlap_naive = len(set(df.loc[idx_train_naive, "client_hash_id"]) & set(df.loc[idx_test_naive, "client_hash_id"]))

rf_naive = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf_naive.fit(X_train_naive, y_train_naive)
naive_scores = rf_naive.predict_proba(X_test_naive)[:, 1]
naive_p50 = precision_at_k(naive_scores, y_test_naive, 50)
naive_auc = roc_auc_score(y_test_naive, naive_scores)

# AFTER — client-grouped split (same as Week 5)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y[train_idx], y[test_idx]
overlap_grouped = len(set(df.iloc[train_idx]["client_hash_id"]) & set(df.iloc[test_idx]["client_hash_id"]))

rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf_grouped.fit(X_train_g, y_train_g)
grouped_scores = rf_grouped.predict_proba(X_test_g)[:, 1]
grouped_p50 = precision_at_k(grouped_scores, y_test_g, 50)
grouped_auc = roc_auc_score(y_test_g, grouped_scores)

comparison = pd.DataFrame({
    "split_type": ["Naive random split (BEFORE)", "Client-grouped split (AFTER)"],
    "client_overlap_train_test": [overlap_naive, overlap_grouped],
    "precision_at_50": [naive_p50, grouped_p50],
    "roc_auc": [naive_auc, grouped_auc]
})
print("=== Before/After: Split Design ===")
display(comparison)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*No feature shows a correlation above 0.9 with the label, so no feature
is trivially encoding the answer. The strongest features are
impressions_fh (0.210) and ctr_fh (0.180) — both moderate, expected
signal strength, not leakage-level correlation. avg_position_fh (-0.141)
is negative as expected (better position = less likely to be declining).
days_since_update_fh (-0.014) is nearly uncorrelated, echoing the Week-4
finding that staleness alone is a weak signal for this label. All
features are built with the _fh suffix convention, confirmed by
construction to come only from the first half of March — never from
clicks_sh, the column the label itself is derived from.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.print("=== Leakage audit: correlation of each feature with the label ===")
for f in features:
    corr = np.corrcoef(X[f], y)[0, 1]
    print(f"{f}: correlation with is_declining = {corr:.3f}")

print("\n=== Checking for suspiciously perfect features ===")
suspicious = [f for f in features if abs(np.corrcoef(X[f], y)[0, 1]) > 0.9]
print("Features with |correlation| > 0.9 (potential leakage):", suspicious if suspicious else "None found")

print("\n=== Confirming label independence from second-half-derived columns ===")
print("Features used:", features)
print("None of these were computed from clicks_sh (the second-half column used to build the label) —")
print("confirmed by construction: all features carry the _fh suffix (first-half only).")


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*Original (too bold): "Random Forest triples the baseline's performance,
proving it's the better approach for prioritizing content refresh."

Rewritten (safe language): "On this client-grouped test split, Random
Forest showed a substantially higher Precision@50 (0.88) than the
Week-4 rule baseline (0.20) — an observed, directional result on one
March 2026 slice. This is decision-support evidence, not proof that
Random Forest is the correct approach in general; performance on other
months or a different label window has not been tested."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.